In [57]:
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
import matplotlib.pyplot as plt
import seaborn as sns; sns.set(style="whitegrid")
import datetime
import yfinance as yf

In [58]:
start = datetime.datetime(2010, 1, 1)
end = datetime.datetime.today().strftime('%Y-%m-%d')

tickers = ['AAPL', 'ADBE', 'ORCL', 'EBAY', 'MSFT', 'QCOM', 'HPQ', 'AMD', 'IBM','NVDA','INTC']


df = yf.download(tickers, start, end)['Close']
df.tail()

[*********************100%***********************]  11 of 11 completed


Ticker,AAPL,ADBE,AMD,EBAY,HPQ,IBM,INTC,MSFT,NVDA,ORCL,QCOM
Date,,,,,,,,,,,
2026-09-22,339.750000,238.250000,623.770020,109.279999,32.040001,231.380005,123.860001,498.000000,228.869995,149.199997,198.270004
2026-09-23,337.019989,240.690002,614.609985,109.089996,32.160000,232.759995,122.599998,500.589996,225.509995,144.559998,197.240005
2026-09-24,335.920013,238.929993,629.260010,107.830002,31.139999,227.059998,127.389999,497.929993,224.580002,139.539993,194.259995
2026-09-25,341.070007,235.470001,630.630005,107.900002,31.299999,225.509995,123.000000,516.169983,225.070007,137.100006,201.970001
2026-09-28,338.399994,231.009995,607.869995,107.320000,31.320000,220.669998,116.029999,509.220001,228.860001,132.600006,187.479996


Now we test our stocks for cointegraion using the pvalue and store the pairs for future use

In [72]:
def find_pairs(df):
    n=df.shape[1]
    ticker=df.keys()
    pairs=[]
    p_values={}
    for i in range(n):
        for j in range(i+1,n):
            p_value=coint(df[ticker[i]],df[ticker[j]])[1]
            if p_value<0.05:
                pairs.append((ticker[i],ticker[j]))
            p_values[(ticker[i],ticker[j])]=p_value
    return pairs,p_values

In [73]:
train_test_split=0.7
split=int(len(df)*train_test_split)
pairs,p_values_train=find_pairs(df[:split])

In [74]:
print(pairs)
print(p_values_train)

[('ADBE', 'MSFT'), ('EBAY', 'ORCL')]
{('AAPL', 'ADBE'): 0.28657473304911646, ('AAPL', 'AMD'): 0.179874735913634, ('AAPL', 'EBAY'): 0.2024263459225154, ('AAPL', 'HPQ'): 0.7008836358563261, ('AAPL', 'IBM'): 1.0, ('AAPL', 'INTC'): 0.8932941449092011, ('AAPL', 'MSFT'): 0.09390908641834544, ('AAPL', 'NVDA'): 0.39955194554064155, ('AAPL', 'ORCL'): 0.10942827388800791, ('AAPL', 'QCOM'): 0.3507036428416157, ('ADBE', 'AMD'): 0.3490924370174143, ('ADBE', 'EBAY'): 0.26862691291924345, ('ADBE', 'HPQ'): 0.7185993661941229, ('ADBE', 'IBM'): 1.0, ('ADBE', 'INTC'): 0.9293645751409523, ('ADBE', 'MSFT'): 0.015733956642697756, ('ADBE', 'NVDA'): 0.883368642061306, ('ADBE', 'ORCL'): 0.060170593278266495, ('ADBE', 'QCOM'): 0.6537817816095085, ('AMD', 'EBAY'): 0.41916843926426167, ('AMD', 'HPQ'): 0.747837008640246, ('AMD', 'IBM'): 1.0, ('AMD', 'INTC'): 0.9509044973750999, ('AMD', 'MSFT'): 0.08206187513042373, ('AMD', 'NVDA'): 0.41913488888663936, ('AMD', 'ORCL'): 0.27504454294411695, ('AMD', 'QCOM'): 0.22240

In [80]:
def trade_ratios(df,pairs,train_test_split=0.7,window_long=60,window_short=5):
    if train_test_split>1 or train_test_split<0 or window_long==0 or window_short==0:
        return 0
    split=int(len(df)*train_test_split)
    results={}
    for i in range (len(pairs)):
        S1=df[pairs[i][0]].iloc[split:]
        S2=df[pairs[i][1]].iloc[split:]
        ratio=S1/S2
        ma_long=ratio.rolling(window=window_long).mean()
        ma_short=ratio.rolling(window=window_short).mean()
        std=ratio.rolling(window=window_long).std()
        zscore=(ma_short-ma_long)/std

        money=0
        countS1=0
        countS2=0

        for j in range(len(ratio)):
            if zscore.iloc[j] < -1:
                money += S1.iloc[j] - S2.iloc[j] * ratio.iloc[j]
                countS1 -= 1
                countS2 += ratio.iloc[j]
            elif zscore.iloc[j] > 1:
                money -= S1.iloc[j] - S2.iloc[j] * ratio.iloc[j]
                countS1 += 1
                countS2 -= ratio.iloc[j]
            elif abs(zscore.iloc[j]) < 0.75:
                money += S1.iloc[j] * countS1 + S2.iloc[j] * countS2
                countS1 = 0
                countS2 = 0
        results[(pairs[i][0],pairs[i][1])]=money
        print(f"{pairs[i]}'s return : {money}")
    return results

In [81]:
results=trade_ratios(df,pairs)

('ADBE', 'MSFT')'s return : 2104.885970670681
('EBAY', 'ORCL')'s return : -1392.8347135046333


In [78]:
def test_coint(pairs,df):
    p_values={}
    for i in range (len(pairs)):
        p_value=coint(df[pairs[i][0]],df[pairs[i][1]])[1]
        if p_value>0.05:
            print(f"{pairs[i]} is not cointegrated on test data,p_value: {p_value}")
        else :
            print(f"{pairs[i]} is cointegrated on test data, p_value: {p_value}")
        p_values[(pairs[i][0],pairs[i][1])]=p_value
    return p_values

In [79]:
p_values_test=test_coint(pairs,df.iloc[split:])

('ADBE', 'MSFT') is not cointegrated on test data,p_value: 0.5546171887818816
('EBAY', 'ORCL') is not cointegrated on test data,p_value: 0.9539993681371491


We see that our pairs were not cointegrated but the pair ADBE and MSFT still made 2k on the test data.